# CCA-F Review Notebook — Agentic Architecture

This notebook is organized by the provided CCA-F task list. Each task includes a concept explanation, runnable Python example, exam summary, and two practice questions.

## Environment Setup

In [ ]:
import json
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional

MODEL = "claude-haiku-4-5-20251001"
print("Environment ready: examples are local simulations, require no API key, and mirror Claude API / Claude Code control flow.")

## D1.1 — Design and implement agentic loops for autonomous task execution

### 📖 Concept Explanation

An agentic loop is the host-side control structure around Claude tool use. The model response tells the host what to do next through `stop_reason`: `tool_use` means Claude is requesting external actions, so the host must execute every `tool_use` block and append matching `tool_result` blocks as the next `user` message; `end_turn` means the model has finished this turn and the final text can be returned to the user. A production loop should also include a maximum-iteration guard, but that guard is a safety fallback, not the normal completion condition.

CCA-F commonly tests these boundaries: do not terminate by parsing phrases such as “done,” do not omit the assistant message that contained the original `tool_use`, do not process only the first tool call, and do not put tool results under the wrong role. The mental model is: Claude chooses the next action; your code executes actions, preserves message order, stores state, and enforces structured termination.

In [ ]:
# Task 1.1: local simulation of an agentic loop.
# Key point: `tool_use` continues the loop; `end_turn` terminates it.
# This example also shows that one model response can request multiple tools.

def execute_tool(name: str, tool_input: dict) -> str:
    if name == "get_customer":
        return json.dumps({"customer_id": "CUST-1001", "verified": True})
    if name == "lookup_order":
        return json.dumps({"order_id": tool_input["order_id"], "amount": 89.0, "eligible": True})
    if name == "process_refund":
        return json.dumps({"refund_id": "RF-7788", "status": "submitted"})
    return json.dumps({"error": f"unknown tool: {name}"})


fake_model_turns = [
    {
        "stop_reason": "tool_use",
        "content": [
            {"type": "tool_use", "id": "toolu_customer", "name": "get_customer", "input": {"email": "a@example.com"}},
            {"type": "tool_use", "id": "toolu_order", "name": "lookup_order", "input": {"order_id": "ORD-9"}},
        ],
    },
    {
        "stop_reason": "tool_use",
        "content": [
            {"type": "tool_use", "id": "toolu_refund", "name": "process_refund", "input": {"order_id": "ORD-9", "amount": 89.0}}
        ],
    },
    {"stop_reason": "end_turn", "content": [{"type": "text", "text": "Customer is verified, the order is refund-eligible, and the refund was submitted."}]},
]

messages = [{"role": "user", "content": "Please refund order ORD-9"}]
max_iterations = 5

for iteration, turn in enumerate(fake_model_turns, start=1):
    if iteration > max_iterations:
        raise RuntimeError("Iteration limit reached: this is a safety guard, not normal completion.")

    print("stop_reason:", turn["stop_reason"])
    if turn["stop_reason"] == "tool_use":
        # Preserve the assistant tool_use, then append user tool_result blocks in the next message.
        messages.append({"role": "assistant", "content": turn["content"]})
        tool_results = []
        for block in turn["content"]:
            result = execute_tool(block["name"], block["input"])
            tool_results.append({"type": "tool_result", "tool_use_id": block["id"], "content": result})
        messages.append({"role": "user", "content": tool_results})
    elif turn["stop_reason"] == "end_turn":
        print(turn["content"][0]["text"])
        break
    else:
        raise ValueError(f"Unhandled stop_reason: {turn['stop_reason']}")

print("message count:", len(messages))
print("last tool result:", messages[-1]["content"][0])

### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Drive the loop with `stop_reason`: continue on `tool_use`, finish on `end_turn`.
- Execute every `tool_use` in the response and return a matching `tool_result` for each `tool_use_id`.
- Preserve Anthropic message order: assistant `tool_use`, then user `tool_result`.
- Add iteration limits, unknown-tool handling, and exception paths as engineering safeguards.

**✗ Anti-patterns / distractors**

- Parsing text such as “done” or “task complete” as the termination signal.
- Appending tool results while omitting the assistant `tool_use` message.
- Executing only the first tool call in a response.
- Treating max iterations as the expected business completion path.

### 🧪 Practice Questions

**Q1.** Claude returns `stop_reason: "tool_use"` and the response `content` contains two `tool_use` blocks. What should the host program do next?

A) Execute both tools, append matching `tool_result` blocks with the correct `tool_use_id`, then call Claude again  
B) Execute only the first tool because Claude can only really use one tool per turn  
C) Return the assistant text directly and wait for user confirmation  
D) Resend the same user message until the model returns `end_turn`

> **Answer: A**  
> **Explanation:** `tool_use` is a structured request for external action; the host must process all tool calls in the response and return results with matching IDs.

**Q2.** Which message order best matches Anthropic tool-use loops?

A) `user` request → assistant `tool_use` → user `tool_result` → next model call  
B) `user` request → user `tool_result` → assistant `tool_use` → next model call  
C) `user` request → execute tool → discard message history → next model call  
D) assistant final text → `tool_result` → `tool_use` → stop

> **Answer: A**  
> **Explanation:** The model first emits `tool_use`; after execution, the host appends the corresponding `tool_result` as a user message. Reversing that order breaks the tool-use context.

## D1.2 — Orchestrate multi-agent systems with coordinator-subagent patterns

### 📖 Concept Explanation

Coordinator-subagent orchestration usually follows a hub-and-spoke pattern. The coordinator is the single entry point and quality-control layer: it decomposes work, selects subagents, passes explicit context, isolates tool permissions, aggregates results, checks conflicts, and identifies coverage gaps. Subagents focus on local assignments; they should not be assumed to share parent conversation history, and they should not form an untraceable peer-to-peer communication mesh.

CCA-F is not testing whether “more agents” sounds advanced. It is testing when decomposition is warranted: broad scope, parallelizable work, different tool permissions, long context, or a need for independent verification. Simple linear tasks are usually better served by one agent or ordinary code. Common distractors reduce the coordinator to a router or skip the synthesis and conflict-checking stage.

In [ ]:
# Task 1.2: hub-and-spoke orchestration.
# The coordinator owns decomposition, explicit context passing, aggregation, and coverage checks.

def subagent(role: str, assignment: str, context: dict) -> dict:
    simulated_findings = {
        "market_research": "Demand clusters around small teams, based on interview notes.",
        "policy_review": "Refund constraints come from refund-policy.md#auto.",
        "risk_review": "High-value refunds require human approval.",
    }
    return {
        "role": role,
        "assignment": assignment,
        "received_context_keys": sorted(context.keys()),
        "finding": simulated_findings[role],
        "confidence": 0.82 if role != "risk_review" else 0.76,
    }


def coordinator(topic: str) -> dict:
    base_context = {
        "topic": topic,
        "required_output": "claim-source mapping",
        "source_policy": "Prefer primary sources",
    }
    assignments = {
        "market_research": "Identify user demand and evidence sources",
        "policy_review": "Check policy terms and eligibility conditions",
        "risk_review": "Find risks that require human escalation",
    }
    required_roles = set(assignments)
    results = [subagent(role, task, base_context) for role, task in assignments.items()]
    returned_roles = {item["role"] for item in results}

    low_confidence = [item["role"] for item in results if item["confidence"] < 0.8]
    return {
        "topic": topic,
        "results": results,
        "coverage_gaps": sorted(required_roles - returned_roles),
        "needs_follow_up": low_confidence,
        "final_summary": "Coordinator merged subtask results and marked low-confidence areas.",
    }


print(json.dumps(coordinator("automatic refund workflow redesign"), indent=2))

### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Use the coordinator for coverage mapping, decomposition, context routing, result aggregation, conflict checks, and failure recovery.
- Give each subagent the smallest complete context and tool set needed for its assignment.
- Use subagents for work that is parallelizable, isolatable, or independently verifiable; keep simple linear work simple.

**✗ Anti-patterns / distractors**

- Sending every request through a full multi-agent pipeline.
- Letting subagents coordinate peer-to-peer, making state and responsibility hard to audit.
- Concatenating outputs without checking coverage gaps, contradictions, or low-confidence results.
- Decomposing too narrowly so entire domains are never assigned.

### 🧪 Practice Questions

**Q1.** A research system covers only visual art and misses music and film. What is the most likely root cause?

A) The coordinator decomposed the research scope too narrowly and lacked a coverage map  
B) The search subagent had too few tokens, so every conclusion is invalid  
C) The synthesis subagent needed direct browser access instead of receiving results  
D) Subagents should communicate peer-to-peer so they can discover missing domains themselves

> **Answer: A**  
> **Explanation:** Coverage gaps usually come from coordinator decomposition and coverage planning, not from asking one subagent to try harder.

**Q2.** Which scenario is the best fit for coordinator-subagent orchestration?

A) A long task requiring policy review, market research, and risk assessment, where each part can be independently checked  
B) A deterministic Celsius-to-Fahrenheit conversion  
C) A small script where every step strictly depends on the previous line of output  
D) A simple question with no tools, no long context, and no quality-control requirement

> **Answer: A**  
> **Explanation:** Multi-agent orchestration is useful for broad, parallel, tool-isolated, or independently verifiable work. Simple deterministic tasks should not be over-designed.

## D1.3 — Configure subagent invocation, context passing, and spawning

### 📖 Concept Explanation

Subagent invocation depends on explicit context and permission boundaries. In Claude Code, the coordinator spawns subagents through the `Task` tool; if `Task` is missing from `allowedTools`, a stronger prompt cannot create a subagent. Every invocation should pass the goal, scope, known facts, sources, output format, forbidden actions, and failure-reporting expectations because subagents do not automatically inherit the parent conversation.

There is also a difference between “passing context” and “leaking the whole chat.” The right pattern is to pass the minimum sufficient context needed to complete the assignment and to configure the child agent with the appropriate tools. Session isolation keeps subagents focused and auditable while reducing irrelevant history and sensitive data exposure.

In [ ]:
# Task 1.3: explicit context injection, Task permission, and session isolation.
# Subagents do not inherit parent conversation history; the coordinator must be allowed to use Task.

def build_task_prompt(goal: str, known_facts: dict, output_schema: dict) -> str:
    return f"""Goal: {goal}

Scope: only determine automatic refund eligibility; do not issue payment or send email.

Known facts explicitly passed by the coordinator:
{json.dumps(known_facts, indent=2)}

Required output format:
{json.dumps(output_schema, indent=2)}

Failure reporting: if evidence is insufficient, return missing_context and explain the gap.
"""


def invoke_subagent(coordinator_config: dict, child_allowed_tools: list[str], prompt: str) -> dict:
    if "Task" not in coordinator_config.get("allowedTools", []):
        raise PermissionError("Coordinator cannot spawn subagents without Task permission.")
    # Local simulation: the child receives a new work package, not the full parent history.
    isolated_session = {"messages": [{"role": "user", "content": prompt}], "allowedTools": child_allowed_tools}
    return {
        "session_message_count": len(isolated_session["messages"]),
        "child_allowed_tools": isolated_session["allowedTools"],
        "inherits_parent_history": False,
    }


coordinator_config = {"allowedTools": ["Task", "Read", "Grep"]}
prompt = build_task_prompt(
    "Determine whether ORD-9 is eligible for automatic refund",
    {
        "customer_id": "CUST-1001",
        "order_id": "ORD-9",
        "policy_source": "refund-policy.md#auto",
        "order_amount": 89.0,
    },
    {"claim": "string", "evidence": "string", "source": "string", "confidence": "number", "missing_context": "list"},
)

print(prompt)
print(json.dumps(invoke_subagent(coordinator_config, ["Read", "Grep"], prompt), indent=2))

### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Include `Task` in the coordinator's `allowedTools` before expecting subagent spawning to work.
- Put goal, scope, known facts, sources, output format, and failure-reporting requirements in the Task prompt.
- Give child agents task-appropriate minimal tools instead of opening every tool by default.
- Treat subagents as isolated sessions, not as automatic continuations of the parent chat.

**✗ Anti-patterns / distractors**

- Assuming subagents can read all parent conversation history.
- Saying only “continue the previous research” without sources, evidence, or output requirements.
- Prompting the coordinator to delegate while withholding the `Task` tool.
- Granting every tool for convenience and breaking the permission boundary.

### 🧪 Practice Questions

**Q1.** How should a coordinator give prior findings to a subagent?

A) Include the goal, sources, excerpts, known facts, and output format in the Task prompt  
B) Assume the subagent reads parent history automatically  
C) Say only “continue the previous research”  
D) Give the subagent every available tool so it can find context by itself

> **Answer: A**  
> **Explanation:** A Task invocation is an isolated work package and needs explicit context for the assignment.

**Q2.** A coordinator prompt says “spawn a research subagent,” but no subagent can be created at runtime. What is the most likely configuration issue?

A) `Task` is missing from the coordinator's `allowedTools`  
B) The subagent did not inherit the parent conversation title  
C) The output schema omitted a `confidence` field  
D) The subagent was not named `researcher`

> **Answer: A**  
> **Explanation:** Subagent spawning depends on `Task` tool permission; prompting cannot bypass tool configuration.

## D1.4 — Implement hook interception mechanisms and session state management

### 📖 Concept Explanation

Hooks are deterministic control points, not another name for prompting. `PreToolUse` runs before a tool executes and can inspect the tool name, arguments, and session state to allow, modify, or veto the call. `PostToolUse` runs after execution and before the result enters model context, which makes it the right place for normalization, audit logging, and sensitive-field handling. Session state stores verified customers, recent tool results, risk flags, and escalation paths so high-risk workflows cannot bypass prerequisites.

CCA-F often contrasts “the model promised to follow the rule” with programmatic enforcement. Identity verification, refund limits, destructive actions, and outbound communication constraints should be enforced through hooks and state machines. Prompts can explain policy, but they should not be the only control layer.

In [ ]:
# Task 1.4: hook interception and session state.
# High-risk operations require programmatic enforcement, not just prompt guidance.

@dataclass
class SupportSession:
    verified_customer_id: Optional[str] = None
    last_tool_results: list[dict] = field(default_factory=list)
    audit_log: list[dict] = field(default_factory=list)


def pre_tool_use(session: SupportSession, tool_name: str, tool_input: dict) -> dict:
    if tool_name in {"lookup_order", "process_refund"} and not session.verified_customer_id:
        return {"action": "veto", "reason": "must_verify_customer_first", "required_tool": "get_customer"}
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 500:
        return {"action": "veto", "reason": "refund_above_auto_limit", "route": "human_approval"}
    return {"action": "allow"}


def post_tool_use(session: SupportSession, tool_name: str, result: dict) -> dict:
    normalized = dict(result)
    if "created_at" in normalized and isinstance(normalized["created_at"], int):
        normalized["created_at"] = datetime.fromtimestamp(normalized["created_at"]).isoformat()
    if "status_code" in normalized:
        normalized["status"] = {1: "active", 2: "suspended"}.get(normalized.pop("status_code"), "unknown")
    session.last_tool_results.append({"tool": tool_name, "result": normalized})
    session.audit_log.append({"tool": tool_name, "normalized": True})
    return normalized


def guarded_tool_call(session: SupportSession, tool_name: str, tool_input: dict, raw_result: dict) -> dict:
    decision = pre_tool_use(session, tool_name, tool_input)
    if decision["action"] == "veto":
        return {"blocked": True, **decision}
    return {"blocked": False, "result": post_tool_use(session, tool_name, raw_result)}


session = SupportSession()
print(guarded_tool_call(session, "process_refund", {"amount": 89}, {"refund_id": "RF-1"}))
session.verified_customer_id = "CUST-1001"
print(guarded_tool_call(session, "process_refund", {"amount": 89}, {"refund_id": "RF-7788", "status_code": 1, "created_at": 1705311000}))
print("audit log:", session.audit_log)

### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Use `PreToolUse` for pre-execution ordering, permission, and parameter checks; veto when needed.
- Use `PostToolUse` to normalize, audit, and redact tool results before they enter model context.
- Store verified facts, tool results, and risk flags in session state.
- Provide recovery paths when blocking, such as “verify customer first” or “route to human approval.”

**✗ Anti-patterns / distractors**

- Relying only on prompts for refund prerequisites.
- Asking the model to interpret status codes, timestamps, or internal error formats by itself.
- Blocking without telling the system which next step or escalation path is required.
- Treating hooks as passive logs rather than execution control points.

### 🧪 Practice Questions

**Q1.** Refunds require verified identity first. What is the most reliable design?

A) Block `process_refund` in `PreToolUse` until `verified_customer_id` exists in session state  
B) Repeat the instruction several times in the system prompt  
C) Ask the model to promise compliance before calling the tool  
D) Make the refund tool description longer

> **Answer: A**  
> **Explanation:** High-risk ordering constraints require programmatic enforcement through hooks and session state.

**Q2.** A tool returns `status_code: 1` and a Unix timestamp. You want the model to see stable, readable, standardized fields. Where should this be handled?

A) `PostToolUse`, before the result enters model context  
B) `PreToolUse`, because it runs before tool execution  
C) The user prompt, because the user knows the display format best  
D) Nowhere; the model should infer all internal encodings

> **Answer: A**  
> **Explanation:** `PostToolUse` is the right point for normalizing, redacting, and auditing tool output. `PreToolUse` is primarily for pre-execution interception.